1. Connect to sqlite 
2. Export to pandas dataframe

In [ ]:
import numpy as np
from sqlalchemy import create_engine
from getpass import getpass
import pandas as pd
import re
import json
import os
from dotenv import load_dotenv
load_dotenv()

pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

password = getpass("Enter password: ")

engine = create_engine(
    "mysql+mysqlconnector://localhost/materials_catalogue",
    connect_args={
        "user": "root",
        "password": password,
    }
)

In [ ]:
# Helper to save doc dict as CSV
def save_docs_to_csv(doc_dict: dict, filepath: str):
    rows = [{COL_04: pid, "text": text} for pid, text in doc_dict.items()]
    pd.DataFrame(rows).to_csv(filepath, index=False)
    print(f"Saved {len(rows)} rows to {filepath}")

# Helper to save chunks list as CSV
def save_chunks_to_csv(chunks: list, filepath: str):
    rows = [{"metadata": json.dumps(c["metadata"]), "text": c["text"]} for c in chunks]
    pd.DataFrame(rows).to_csv(filepath, index=False)
    print(f"Saved {len(rows)} rows to {filepath}")

def is_empty(val):
    """Treat NULL, empty string, 'ND', 'MND' as missing."""
    if val is None:
        return True
    s = str(val).strip().upper()
    return s in ("", "NULL", "ND", "MND", "N/A", "-")

# Save all
# save_chunks_to_csv(name_chunks, "name_chunks.csv")

### Table 'epd_insights'

In [ ]:
TABLE_01 = os.getenv("TABLE_01")
TABLE_02 = os.getenv("TABLE_02")
TABLE_03 = os.getenv("TABLE_03")
TABLE_04 = os.getenv("TABLE_04")
TABLE_05 = os.getenv("TABLE_05")
TABLE_06 = os.getenv("TABLE_06")
TABLE_07 = os.getenv("TABLE_07")

with engine.connect() as connection:
    df_01 = pd.read_sql(f"SELECT * FROM {TABLE_01}", connection)
    df_02 = pd.read_sql(f"SELECT * FROM {TABLE_02}", connection)
    df_03 = pd.read_sql(f"SELECT * FROM {TABLE_03}", connection)
    df_04 = pd.read_sql(f"SELECT * FROM {TABLE_04}", connection)
    df_05 = pd.read_sql(f"SELECT * FROM {TABLE_05}", connection)
    df_06 = pd.read_sql(f"SELECT * FROM {TABLE_06}", connection)
    df_07 = pd.read_sql(f"SELECT * FROM {TABLE_07}", connection)

# print("df_02 columns:", df_02.columns.tolist())
# print("df_02 columns count:", len(df_02.columns.tolist()))
# print("df_03 columns:", df_03.columns.tolist())
# print("df_04 columns:", df_04.columns.tolist())
# print("df_05 columns:", df_05.columns.tolist())

In [ ]:
COL_01 = os.getenv("COL_01")
COL_02 = os.getenv("COL_02")
COL_03 = os.getenv("COL_03")
COL_04 = os.getenv("COL_04")
COL_05 = os.getenv("COL_05")
COL_06 = os.getenv("COL_06")

INDICATOR_MAP = json.loads(os.getenv("INDICATOR_MAP"))

df_03[COL_05] = df_03[COL_01].map(INDICATOR_MAP).fillna(df_03[COL_01])

def build_combined_df(df_01, df_03):

    rows_before = len(df_01)

    # Replace empty strings with NaN before casting
    df_01[COL_02] = df_01[COL_02].replace("", pd.NA).astype("Int64")

    df = df_01.merge(
        df_03[[COL_03, COL_05]].rename(columns={COL_05: COL_06}),
        left_on=COL_02,
        right_on=COL_03,
        how="left"
    )

    # Drop rows with no indicator name
    df = df.dropna(subset=[COL_06])
    rows_after = len(df)
    print(f"Rows before: {rows_before} | Rows after: {rows_after} | Dropped: {rows_before - rows_after}")

    return df

combined_df = build_combined_df(df_01, df_03)

# Verify
print(combined_df[[COL_04, COL_06]].tail(5))
print(combined_df.columns)

In [ ]:
# 1. Configuration

COL_07 = os.getenv("COL_07")
COL_08 = os.getenv("COL_08")
COL_09 = os.getenv("COL_09")
COL_10 = os.getenv("COL_10")
COL_11 = os.getenv("COL_11")
COL_12 = os.getenv("COL_12")
COL_13 = os.getenv("COL_13")
COL_14 = os.getenv("COL_14")

LOWER_IS_BETTER = [INDICATOR_MAP[c] for c in json.loads(os.getenv("LOWER_IS_BETTER_CODES"))]
HIGHER_IS_BETTER = [INDICATOR_MAP[c] for c in json.loads(os.getenv("HIGHER_IS_BETTER_CODES"))]

INDICATOR_COLS = LOWER_IS_BETTER + HIGHER_IS_BETTER

LIFECYCLE_STAGES = [COL_09, COL_10, COL_11, COL_12, COL_13, COL_14]

# 2. Build material lookup

product_to_material = (
    df_06[[COL_04, COL_07]]
    .merge(
        df_05[[COL_03, COL_01]].rename(columns={COL_01: COL_08}),
        left_on=COL_07,
        right_on=COL_03,
        how="left"
    )
    .drop(columns=[COL_03])
    .drop_duplicates(subset=COL_04)
)

# 3. Join material name onto insights 

df = (
    combined_df
    .merge(df_07[[COL_03]].rename(columns={COL_03: COL_04}),
           on=COL_04, how="left")
    .merge(product_to_material[[COL_04, COL_08]],
           on=COL_04, how="left")
)

# 4. Filter to relevant indicators and cast stage columns to numeric 

df = df[df[COL_06].isin(INDICATOR_COLS)].copy()

for col in LIFECYCLE_STAGES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 5. Flag rows where ALL stage values are 0 for that indicator

df["all_stages_zero"] = (df[LIFECYCLE_STAGES] == 0).all(axis=1)

# 6. Assign impact labels per indicator+stage within material TABLE_26

def assign_labels(series, all_zero_mask, lower_is_better=True):
    labels = pd.Series(index=series.index, dtype="object")
    valid = series.dropna()
    valid = valid[~all_zero_mask.reindex(valid.index, fill_value=False)]

    if len(valid) < 3:
        labels[valid.index] = "moderate"
        return labels

    t33 = valid.quantile(0.33)
    t66 = valid.quantile(0.66)

    if lower_is_better:
        conditions = [valid <= t33, (valid > t33) & (valid <= t66), valid > t66]
    else:
        conditions = [valid >= t66, (valid >= t33) & (valid < t66), valid < t33]

    for cond, label in zip(conditions, ["low", "moderate", "high"]):
        labels[valid[cond].index] = label

    return labels


result_df = df.copy()

for stage in LIFECYCLE_STAGES:
    for indicator in INDICATOR_COLS:
        lower_is_better = indicator in LOWER_IS_BETTER
        label_col = f"{stage}_{indicator}_impact"

        mask = df[COL_06] == indicator

        result_df.loc[mask, label_col] = (
            df[mask]
            .groupby(COL_08)[stage]
            .transform(lambda s: assign_labels(
                s,
                all_zero_mask=df.loc[s.index, "all_stages_zero"],
                lower_is_better=lower_is_better
            ))
        )

result_df.sample(10)

In [ ]:
# ----- 1. Configuration -----

LOWER_IS_BETTER = [INDICATOR_MAP[c] for c in json.loads(os.getenv("LOWER_IS_BETTER_CODES"))]
HIGHER_IS_BETTER = [INDICATOR_MAP[c] for c in json.loads(os.getenv("HIGHER_IS_BETTER_CODES"))]

INDICATOR_COLS = LOWER_IS_BETTER + HIGHER_IS_BETTER
LIFECYCLE_STAGES = [COL_09, COL_10, COL_11, COL_12, COL_13, COL_14]

_indicator_narratives_raw = json.loads(os.getenv("INDICATOR_NARRATIVES"))
INDICATOR_NARRATIVES = {
    INDICATOR_MAP[code]: tuple(value) for code, value in _indicator_narratives_raw.items()
}

_phase_narratives_list = json.loads(os.getenv("PHASE_NARRATIVES_LIST"))
PHASE_NARRATIVES = dict(zip(LIFECYCLE_STAGES, _phase_narratives_list))

# Maps severity levels to descriptive adverbs used in natural sentences.
LEVEL_ADVERBS = {
    "low":      "low",
    "moderate": "moderate",
    "high":     "high",
}

# ----- 2. Build material lookup -----

product_to_material = (
    df_06[[COL_04, COL_07]]
    .merge(
        df_05[[COL_03, COL_01]].rename(columns={COL_01: COL_08}),
        left_on=COL_07,
        right_on=COL_03,
        how="left"
    )
    .drop(columns=[COL_03])
    .drop_duplicates(subset=COL_04)
)

# ----- 3. Join material name onto insights -----

df = (
    combined_df
    .merge(df_07[[COL_03]].rename(columns={COL_03: COL_04}),
           on=COL_04, how="left")
    .merge(product_to_material[[COL_04, COL_08]],
           on=COL_04, how="left")
)

# ----- 4. Filter to relevant indicators and cast stage columns to numeric -----

df = df[df[COL_06].isin(INDICATOR_COLS)].copy()

for col in LIFECYCLE_STAGES:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ----- 5. Flag rows where ALL stage values are 0 -----

df["all_stages_zero"] = (df[LIFECYCLE_STAGES] == 0).all(axis=1)

# ----- 6. Assign impact labels per indicator+stage within material TABLE_26 -----

def assign_labels(series, all_zero_mask, lower_is_better=True):
    labels = pd.Series(index=series.index, dtype="object")
    valid = series.dropna()
    valid = valid[~all_zero_mask.reindex(valid.index, fill_value=False)]

    if len(valid) < 3:
        labels[valid.index] = "moderate"
        return labels

    t33 = valid.quantile(0.33)
    t66 = valid.quantile(0.66)

    if lower_is_better:
        conditions = [valid <= t33, (valid > t33) & (valid <= t66), valid > t66]
    else:
        conditions = [valid >= t66, (valid >= t33) & (valid < t66), valid < t33]

    for cond, label in zip(conditions, ["low", "moderate", "high"]):
        labels[valid[cond].index] = label

    return labels


result_df = df.copy()

for stage in LIFECYCLE_STAGES:
    for indicator in INDICATOR_COLS:
        lower_is_better = indicator in LOWER_IS_BETTER
        label_col = f"{stage}_{indicator}_impact"
        mask = df[COL_06] == indicator

        result_df.loc[mask, label_col] = (
            df[mask]
            .groupby(COL_08)[stage]
            .transform(lambda s: assign_labels(
                s,
                all_zero_mask=df.loc[s.index, "all_stages_zero"],
                lower_is_better=lower_is_better
            ))
        )

# ----- 7. Serialize into chunks using percentile-derived labels -----

def serialize_product_chunks(product_id: str, rows: pd.DataFrame) -> list[dict]:
    chunks = []

    for _, row in rows.iterrows():
        indicator = str(row[COL_06]).strip()
        narrative_entry = INDICATOR_NARRATIVES.get(indicator)

        if narrative_entry:
            concept, synonyms = narrative_entry
        else:
            concept  = indicator.rsplit("[", 1)[0].strip()
            synonyms = None

        lines = []
        synonym_clause = f"{indicator} (also referred to as {synonyms})" if synonyms else ""
        lines.append(f"{synonym_clause}:")

        stage_levels = {}
        for stage in LIFECYCLE_STAGES:
            val = row.get(stage)
            if is_empty(val):
                continue

            label_col = f"{stage}_{indicator}_impact"
            level = row.get(label_col)
            if not level or level not in LEVEL_ADVERBS:  # skip if no label
                continue

            phase = PHASE_NARRATIVES.get(stage, stage)
            adverb = LEVEL_ADVERBS[level]
            stage_levels[stage] = level
            lines.append(f"{adverb} {concept} {phase}.")

        if not stage_levels:  # skip chunk entirely if no labelled stages
            continue

        chunks.append({
            "text": " ".join(lines),
            "metadata": {
                COL_04:    product_id,
                "material_name": row.get(COL_08),
                "doc_type":      f"{indicator}_epd",
                "concept":       concept,
            }
        })

    return chunks


def build_all_chunks(df: pd.DataFrame, product_col: str = None) -> list[dict]:
    product_col = product_col or COL_04
    all_chunks = []
    for pid, group in df.groupby(product_col):
        chunks = serialize_product_chunks(pid, group)
        all_chunks.extend(chunks)
    return all_chunks


epd_chunks = build_all_chunks(result_df)

print(f"Total chunks: {len(epd_chunks)}")
for chunk in epd_chunks[:3]:
    print("metadata:", chunk["metadata"])
    print("\n", chunk["text"])

In [ ]:
save_chunks_to_csv(epd_chunks, "processed_chunks/epd_chunks.csv")

### Table 'product_materials'

In [ ]:
TABLE_17 = os.getenv("TABLE_17")

with engine.connect() as connection:
    df_06 = pd.read_sql(f"SELECT * FROM {TABLE_06}", connection)
    df_05 = pd.read_sql(f"SELECT * FROM {TABLE_05}", connection)
    df_04 = pd.read_sql(f"SELECT * FROM {TABLE_04}", connection)
    df_17 = pd.read_sql(f"SELECT * FROM {TABLE_17}", connection)

print("df_06 columns:", df_06.columns.tolist())
print("df_06 columns count:", len(df_06.columns.tolist()))
print("df_05 columns:", df_05.columns.tolist())
print("df_04 columns:", df_04.columns.tolist())
print("df_17 columns:", df_17.columns.tolist())

In [11]:
COL_15 = os.getenv("COL_15")
COL_16 = os.getenv("COL_16")
COL_17 = os.getenv("COL_17")
COL_18 = os.getenv("COL_18")
COL_19 = os.getenv("COL_19")
COL_20 = os.getenv("COL_20")
COL_21 = os.getenv("COL_21")
COL_22 = os.getenv("COL_22")
COL_23 = os.getenv("COL_23")
COL_24 = os.getenv("COL_24")
COL_25 = os.getenv("COL_25")
COL_26 = os.getenv("COL_26")
COL_27 = os.getenv("COL_27")
COL_28 = os.getenv("COL_28")
COL_29 = os.getenv("COL_29")
COL_30 = os.getenv("COL_30")
COL_31 = os.getenv("COL_31")
COL_32 = os.getenv("COL_32")
COL_33 = os.getenv("COL_33")
COL_34 = os.getenv("COL_34")
COL_35 = os.getenv("COL_35")
COL_36 = os.getenv("COL_36")

TIER_PRIMARY = os.getenv("TIER_PRIMARY")
TIER_SECONDARY = os.getenv("TIER_SECONDARY")
TIER_TERTIARY = os.getenv("TIER_TERTIARY")

V_OPTIONS = json.loads(os.getenv("V_OPTIONS_MAP"))


def build_materials_combined_df(df_06, df_05, df_17, df_04):

    mat_cols = [COL_03, COL_01, COL_15, COL_16, COL_17,
                COL_18, COL_19, COL_20, COL_21,
                COL_22, COL_23, COL_24]

    # Resolve density_unit ID to unit name before anything else
    units = df_04[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).rename(
        columns={COL_01: COL_25}
    )
    units[COL_03] = units[COL_03].astype(str)

    mat = df_05[mat_cols].drop_duplicates(subset=[COL_03]).copy()
    mat = mat.merge(units, left_on=COL_22, right_on=COL_03, how="left")

    # Combine density + resolved unit name
    mat[COL_26] = mat.apply(
        lambda row: (
            f"{row[COL_21]} {row[COL_25]}".strip()
            if pd.notna(row[COL_21]) and not is_empty(str(row[COL_21]))
            and pd.notna(row[COL_25]) and not is_empty(str(row[COL_25]))
            else str(row[COL_21]).strip() if pd.notna(row[COL_21]) else None
        ),
        axis=1
    )

    # Drop the extra id column from units merge
    mat = mat.drop(columns=["id_y"]).rename(columns={"id_x": COL_03})

    countries = df_17[[COL_27, COL_01]].drop_duplicates(subset=[COL_27]).rename(
        columns={COL_27: COL_28, COL_01: COL_29}
    )

    def merge_material(df, tier):
        merged = df.merge(
            mat.rename(columns={
                COL_03: f"{tier}_{COL_33}",
                COL_01: f"{tier}_{COL_08}",
                COL_15: f"{tier}_{COL_34}",
                COL_16: f"{tier}_{COL_16}",
                COL_17: f"{tier}_{COL_17}",
                COL_18: f"{tier}_{COL_18}",
                COL_19: f"{tier}_{COL_19}",
                COL_20: f"{tier}_{COL_20}",
                COL_26: f"{tier}_{COL_26}",
                COL_23: f"{tier}_{COL_23}",
            }),
            left_on=f"{tier}_{COL_30}",
            right_on=f"{tier}_{COL_33}",
            how="left"
        ).merge(
            countries.rename(columns={
                COL_28: f"{tier}_{COL_28}",
                COL_29: f"{tier}_{COL_29}",
            }),
            left_on=f"{tier}_{COL_31}",
            right_on=f"{tier}_{COL_28}",
            how="left"
        )
        return merged

    df = df_06.copy()
    df = df[df[COL_07].notna() & (df[COL_07] != 0)]

    df = merge_material(df, TIER_PRIMARY)
    df = merge_material(df, TIER_SECONDARY)
    df = merge_material(df, TIER_TERTIARY)
    df = df.dropna(subset=[f"{TIER_PRIMARY}_{COL_08}"])

    df = df.rename(columns={COL_36: f"{TIER_PRIMARY}_{COL_32}"})

    return df


def is_empty(val):
    """Treat NULL, empty string, 'ND', 'MND' as missing."""
    if val is None:
        return True
    s = str(val).strip().upper()
    return s in ("", "NULL", "ND", "MND", "N/A", "-")


def classify_recyclability(value) -> str | None:
    try:
        rec = float(value)
        if rec <= 0:
            return None
        elif rec < 25:
            return "low recyclability"
        elif rec < 75:
            return "moderate recyclability"
        elif rec < 90:
            return "high recyclability"
        else:
            return "very high recyclability"
    except (ValueError, TypeError):
        return None

def classify_durability(value: str) -> str | None:
    if is_empty(str(value)) or pd.isna(value):
        return None

    numbers = re.findall(r"\d+", str(value))
    if not numbers:
        return None

    years = sum(float(n) for n in numbers) / len(numbers)

    if years < 25:
        return f"short service life ({value.strip()})"
    elif years < 50:
        return f"moderate service life ({value.strip()})"
    elif years <= 100:
        return f"long service life ({value.strip()})"
    else:
        return f"very long service life ({value.strip()})"

def clean_versatility(text: str) -> str | None:
    if is_empty(str(text)) or pd.isna(text):
        return None

    codes_found = re.findall(r"\(?([a-eA-E])\)", str(text))
    codes_found = list(dict.fromkeys([c.lower() for c in codes_found]))

    if codes_found:
        descriptions = [V_OPTIONS[c] for c in codes_found if c in V_OPTIONS]
        if descriptions:
            return "this material " + "; ".join(descriptions)

    cleaned = re.sub(r"\([a-eA-E]\)\s*", "", str(text))
    cleaned = re.sub(r"\s+", " ", cleaned).strip().rstrip(".")
    return cleaned if cleaned else None

def compute_material_density_percentiles(df_05: pd.DataFrame) -> dict[str, tuple[float, float]]:
    """
    Compute p25/p75 density thresholds per parent category,
    reading directly from df_05 where density lives natively.
    """
    percentiles = {}

    for parent, group in df_05.groupby(COL_15):
        numeric = (
            pd.to_numeric(group[COL_21], errors="coerce")
            .dropna()
        )
        if len(numeric) > 4:
            percentiles[parent] = (
                numeric.quantile(0.25),
                numeric.quantile(0.75)
            )

    return percentiles

def classify_density(density_with_unit: str, parent_name: str, percentiles: dict) -> str | None:
    """
    Classify density as low/moderate/high relative to other materials
    in the same parent category.
    """
    if is_empty(str(density_with_unit)) or density_with_unit is None:
        return None
    if parent_name not in percentiles:
        return None

    m = re.match(r"[\d.]+", str(density_with_unit))
    if not m:
        return None

    val = float(m.group())
    p25, p75 = percentiles[parent_name]

    if val < p25:
        return "low"
    elif val > p75:
        return "high"
    else:
        return "moderate"


def serialize_material_chunks(
    product_id,
    rows: pd.DataFrame,
    density_percentiles: dict = None
) -> list[dict]:
    chunks = []

    for _, row in rows.iterrows():

        materials_summary = []
        for tier in [TIER_PRIMARY, TIER_SECONDARY, TIER_TERTIARY]:
            name    = row.get(f"{tier}_{COL_08}")
            content = row.get(f"{tier}_{COL_32}")
            if is_empty(str(name)) or pd.isna(name):
                continue
            try:
                if not is_empty(str(content)) and pd.notna(content) and float(content) > 0:
                    materials_summary.append(f"{name} ({float(content):.0f}%)")
                else:
                    materials_summary.append(str(name))
            except (ValueError, TypeError):
                materials_summary.append(str(name))

        if not materials_summary:
            continue

        lines = []

        if len(materials_summary) == 1:
            lines.append(f"This product is made from {materials_summary[0]}.")
        else:
            lines.append(f"This product is made from: {', '.join(materials_summary)}.")
        lines.append("")

        for tier in [TIER_PRIMARY, TIER_SECONDARY, TIER_TERTIARY]:
            name    = row.get(f"{tier}_{COL_08}")
            content = row.get(f"{tier}_{COL_32}")

            if is_empty(str(name)) or pd.isna(name):
                continue

            parent        = row.get(f"{tier}_{COL_34}")
            country_name  = row.get(f"{tier}_{COL_29}")
            breeam        = row.get(f"{tier}_{COL_16}")
            organic       = row.get(f"{tier}_{COL_17}")
            durability_val = row.get(f"{tier}_{COL_18}")
            durability_dd  = row.get(f"{tier}_{COL_35}")
            versatality   = row.get(f"{tier}_{COL_19}")
            recyclability = row.get(f"{tier}_{COL_20}")
            density       = row.get(f"{tier}_{COL_26}")
            thickness     = row.get(f"{tier}_{COL_23}")

            intro = f"{tier.capitalize()} material: {name}"
            if not is_empty(str(parent)) and pd.notna(parent):
                intro += f", belonging to the {parent} category"
            try:
                if not is_empty(str(content)) and pd.notna(content) and float(content) > 0:
                    intro += f", comprising {float(content):.0f}% of the product"
            except (ValueError, TypeError):
                pass
            if not is_empty(str(country_name)) and pd.notna(country_name):
                intro += f", sourced from {country_name}"
            intro += "."
            lines.append(intro)

            classification_parts = []
            if not is_empty(str(organic)) and pd.notna(organic):
                classification_parts.append(organic.lower())
            if not is_empty(str(breeam)) and pd.notna(breeam):
                classification_parts.append(f"falls under BREEAM category {breeam}")
            if classification_parts:
                lines.append(f"It is {' and '.join(classification_parts)}.")

            physical_parts = []
            if not is_empty(str(density)) and pd.notna(density) and density_percentiles:
                band = classify_density(density, parent, density_percentiles)
                if band:
                    physical_parts.append(f"{band} density ({density})")
                else:
                    physical_parts.append(f"density of {density}")
            if not is_empty(str(thickness)) and pd.notna(thickness):
                physical_parts.append(f"thickness of {thickness}")
            if physical_parts:
                lines.append(f"{', '.join(physical_parts)}.")

            eol_parts = []
            rec_level = classify_recyclability(recyclability)
            if rec_level:
                eol_parts.append(rec_level)

            versatility_clean = clean_versatility(versatality)
            if versatility_clean:
                eol_parts.append(versatility_clean)

            if eol_parts:
                lines.append(f"{'; '.join(eol_parts)}.")

            decision_clean = str(durability_dd).strip() if pd.notna(durability_dd) else ""

            if not decision_clean or "no evidence" in decision_clean.lower():
                dur_level = None
            elif "doesn't meet" in decision_clean.lower() or "does not meet" in decision_clean.lower():
                dur_level = "not durable"
            elif decision_clean.lower().startswith("yes"):
                dur_level = classify_durability(durability_val)
            else:
                dur_level = None

            if dur_level:
                lines.append(f"{dur_level}.")

            lines.append("")

        chunks.append({
            "text": " ".join(lines),
            "metadata": {
                COL_04:  str(product_id),
                "doc_type":    "materials",
                "materials":   ", ".join(materials_summary),
            }
        })

    return chunks

def build_all_material_chunks(
    df: pd.DataFrame,
    df_05: pd.DataFrame,
    product_col: str = None
) -> list[dict]:
    product_col = product_col or COL_04
    density_percentiles = compute_material_density_percentiles(df_05)

    all_chunks = []
    for pid, group in df.groupby(product_col):
        chunks = serialize_material_chunks(pid, group, density_percentiles)
        all_chunks.extend(chunks)
    return all_chunks

In [ ]:
# 1. Build the merged dataframe
df = build_materials_combined_df(df_06, df_05, df_17, df_04)

# 2. Compute density percentiles from materials_df directly
density_percentiles = compute_material_density_percentiles(df_05)

# 3. Build chunks
material_chunks = build_all_material_chunks(df, df_05)

# 4. Print a sample
for chunk in material_chunks[:2]:
    print("--- METADATA ---")
    print(chunk["metadata"])
    print("--- TEXT ---")
    print(chunk["text"])
    print()

In [ ]:
save_chunks_to_csv(material_chunks, "processed_chunks/material_chunks.csv")

### Table 'product_certifications'

In [ ]:
with engine.connect() as connection:
    df_18 = pd.read_sql("SELECT * FROM TABLE_08", connection)
    df_19 = pd.read_sql("SELECT * FROM TABLE_18", connection)

print("df_18 columns:", df_18.columns.tolist())
print("df_18 columns:", len(df_18.columns.tolist()))
print("df_19 columns:", df_19.columns.tolist())

In [16]:
def build_certifications_combined_df(df_18, df_19):

    cert_cols = [COL_03, COL_01, "leed_score", "voc_emissions", "formeldehyde_emissions"]
    certs = df_19[cert_cols].drop_duplicates(subset=[COL_03]).copy()

    df = df_18.merge(
        certs.rename(columns={
            COL_03:   "cert_id",
            COL_01: "certification_name",
        }),
        left_on="certification_id",
        right_on="cert_id",
        how="left"
    )

    rows_before = len(df)
    df = df.dropna(subset=["certification_name"])
    rows_after = len(df)
    print(f"Rows before: {rows_before} | Rows after: {rows_after} | Dropped: {rows_before - rows_after}")

    return df

def serialize_certification_chunks(product_id, rows: pd.DataFrame) -> list[dict]:
    chunks = []
    lines  = []

    cert_names = []

    for _, row in rows.iterrows():
        name = row.get("certification_name")
        if is_empty(str(name)) or pd.isna(name):
            continue

        cert_names.append(name)

        exp_date = row.get("exp_date")
        leed     = row.get("leed_score")
        voc      = row.get("voc_emissions")
        form     = row.get("formeldehyde_emissions")

        cert_line = [f"{name}"]

        if not is_empty(str(exp_date)) and pd.notna(exp_date):
            cert_line.append(f", valid until {exp_date}")
        if not is_empty(str(leed)) and pd.notna(leed):
            cert_line.append(f", contributes to LEED credits: {str(leed).strip()}")
        if not is_empty(str(voc)) and pd.notna(voc):
            cert_line.append(f", meets VOC emissions threshold of {float(voc):.1f}")
        if not is_empty(str(form)) and pd.notna(form):
            cert_line.append(f", meets formaldehyde emissions threshold of {float(form):.1f}")

        lines.append("".join(cert_line) + ".")

    if not cert_names:
        return chunks

    # Summary line + all cert details in one chunk
    summary = f"This product holds the following certifications:"
    full_text = summary + " ".join(lines)

    chunks.append({
        "text": full_text,
        "metadata": {
            COL_04:          str(product_id),
            "doc_type":            "certifications",
            "certification_names": ", ".join(cert_names),
        }
    })

    return chunks


def build_all_certification_chunks(df: pd.DataFrame, product_col: str = COL_04) -> list[dict]:
    all_chunks = []
    for pid, group in df.groupby(product_col):
        chunks = serialize_certification_chunks(pid, group)
        all_chunks.extend(chunks)
    return all_chunks

In [ ]:
certifications_combined_df = build_certifications_combined_df(df_18, df_19)
certification_chunks = build_all_certification_chunks(certifications_combined_df)

print(f"Total certification chunks: {len(certification_chunks)}")
print(f"\n--- METADATA ---")
print(certification_chunks[98]["metadata"])
print(f"--- TEXT ---")
print(certification_chunks[98]["text"])

In [ ]:
save_chunks_to_csv(certification_chunks, "processed_chunks/certification_chunks.csv")

### Table 'product_testing_standards'

In [ ]:
with engine.connect() as connection:
    df_20 = pd.read_sql("SELECT * FROM TABLE_09", connection)
    df_21 = pd.read_sql("SELECT * FROM TABLE_19", connection)

print("prod_testing_standards_df columns:", df_20.columns.tolist())
print("testing_standards_df columns:", df_21.columns.tolist())

In [21]:
def build_testing_standards_combined_df(df_20, df_21):

    standards = df_21[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).copy()

    df = df_20.merge(
        standards.rename(columns={
            COL_03:   "standard_id",
            COL_01: "testing_standard_name",
        }),
        left_on="certification_id",
        right_on="standard_id",
        how="left"
    )

    rows_before = len(df)
    df = df.dropna(subset=["testing_standard_name"])
    rows_after = len(df)
    print(f"Rows before: {rows_before} | Rows after: {rows_after} | Dropped: {rows_before - rows_after}")

    return df


def build_all_testing_standard_chunks(df: pd.DataFrame, product_col: str = COL_04) -> list[dict]:
    all_chunks = []

    for pid, group in df.groupby(product_col):
        standard_names = [
            str(row["testing_standard_name"])
            for _, row in group.iterrows()
            if not is_empty(str(row.get("testing_standard_name"))) and pd.notna(row.get("testing_standard_name"))
        ]

        if not standard_names:
            continue

        summary  = f"This product complies with the following standards: {', '.join(standard_names)}."

        all_chunks.append({
            "text": summary,
            "metadata": {
                COL_04:       str(pid),
                "doc_type":         "testing",
                "standard_names":   ", ".join(standard_names),
            }
        })

    return all_chunks

In [ ]:
standards_df = build_testing_standards_combined_df(df_20, df_21)
all_standard_chunks = build_all_testing_standard_chunks(standards_df)

print(f"Total standards chunks: {len(all_standard_chunks)}")
print(f"\n--- METADATA ---")
print(all_standard_chunks[98]["metadata"])
print(f"--- TEXT ---")
print(all_standard_chunks[98]["text"])

In [ ]:
save_chunks_to_csv(all_standard_chunks, "processed_chunks/standards_chunks.csv")

### Table 'product_other_tags'

In [ ]:
with engine.connect() as connection:
    df_22 = pd.read_sql("SELECT * FROM TABLE_10", connection)
    df_23 = pd.read_sql("SELECT * FROM TABLE_20", connection)

print("df_22 columns:", df_22.columns.tolist())
print("other_tages_df columns:", df_23.columns.tolist())

In [25]:
def build_other_tags_combined_df(df_22, df_23):

    tags = df_23[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).copy()

    df = df_22.merge(
        tags.rename(columns={
            COL_03:   "tag_id",
            COL_01: "tag_name",
        }),
        left_on="certification_id",
        right_on="tag_id",
        how="left"
    )

    rows_before = len(df)
    df = df.dropna(subset=["tag_name"])
    rows_after = len(df)
    print(f"Rows before: {rows_before} | Rows after: {rows_after} | Dropped: {rows_before - rows_after}")

    return df


def serialize_other_tags_chunks(product_id, rows: pd.DataFrame) -> list[dict]:
    chunks = []

    tag_names = []
    for _, row in rows.iterrows():
        name = row.get("tag_name")
        if not is_empty(str(name)) and pd.notna(name):
            tag_names.append(name)

    if not tag_names:
        return chunks

    chunks.append({
        "text": f"{', '.join(tag_names)}.",
        "metadata": {
            COL_04: product_id,
            "doc_type": "tags",
            "tags": tag_names,
        }
    })

    return chunks


def build_all_other_tag_chunks(df: pd.DataFrame, product_col: str = COL_04) -> list[dict]:
    all_chunks = []

    for pid, group in df.groupby(product_col):
        chunks = serialize_other_tags_chunks(pid, group)
        all_chunks.extend(chunks)

    return all_chunks

In [ ]:
tags_df = build_other_tags_combined_df(df_22, df_23)
all_tag_chunks = build_all_other_tag_chunks(tags_df)

for chunk in all_tag_chunks[:2]:
    print("--- METADATA ---")
    print(chunk["metadata"])
    print("--- TEXT ---")
    print(chunk["text"])
    print()

In [ ]:
save_chunks_to_csv(all_tag_chunks, "processed_chunks/tag_chunks.csv")

### Table 'product_score'

In [ ]:
with engine.connect() as connection:
    df_24 = pd.read_sql("SELECT * FROM TABLE_11", connection)
    df_25 = pd.read_sql("SELECT * FROM TABLE_17", connection)

print("df_24 columns:", df_24.columns.tolist())
print("df_25 columns:", df_25.columns.tolist())

In [29]:
def build_product_scores_combined_df(df_24, df_25):

    countries = df_25[[COL_27, COL_01]].drop_duplicates(subset=[COL_27]).rename(
        columns={COL_27: COL_28, COL_01: COL_29}
    )

    df = df_24.merge(
        countries,
        left_on=COL_28,
        right_on=COL_28,
        how="left"
    )

    df = df.drop_duplicates(subset=[COL_04, COL_28])

    duplicated_pids = df[df.duplicated(subset=[COL_04], keep=False)][COL_04].unique()
    df = df[~((df[COL_04].isin(duplicated_pids)) & (df[COL_28] == 0))]

    return df


def describe_score(score, baseline, dimension):
    try:
        score = float(score)
        baseline = float(baseline)
    except (ValueError, TypeError):
        return None

    if score == -1 or pd.isna(score):
        return None

    if pd.isna(baseline) or baseline == 0:
        return None

    diff_pct = ((score - baseline) / baseline) * 100

    if diff_pct > 50:
        relative = f"exceptional {dimension} performance"
    elif diff_pct > 20:
        relative = f"strong {dimension} performance"
    elif diff_pct > 0:
        relative = f"good {dimension} performance"
    elif diff_pct > -20:
        relative = f"below average {dimension} performance"
    else:
        relative = f"poor {dimension} performance"

    return relative

def describe_benefit(benefit, dimension):
    try:
        benefit = float(benefit)
    except (ValueError, TypeError):
        return None
    
    if pd.isna(benefit):
        return None
    if benefit > 0:
        return f"positive {dimension} benefit"
    elif benefit < 0:
        return f"negative {dimension} benefit"
    return None

DIMENSION_OPTIONS = json.loads(os.getenv("DIMENSION_OPTIONS_MAP"))

def serialize_product_score_chunks(product_id, rows: pd.DataFrame) -> list[dict]:
    chunks = []

    for _, row in rows.iterrows():
        country = row.get(COL_29)
        country_str = f"in {country}" if not is_empty(str(country)) and pd.notna(country) else "globally"

        dimension_results = {}
        for dim, label in DIMENSION_OPTIONS.items():
            score    = row.get(f"{dim}_score")
            baseline = row.get(f"{dim}_baseline")
            desc     = describe_score(score, baseline, label)
            if desc:
                dimension_results[label] = (desc, float(score), float(baseline))

        if not dimension_results:
            continue

        # ---------------------------------------------------------------- #
        # Metadata — structured fields for filtering                       #
        # ---------------------------------------------------------------- #
        metadata = {
            COL_04: product_id,
            "doc_type": "scores",
            "country":    country if not is_empty(str(country)) and pd.notna(country) else None,
        }

        # Performance level per dimension — for filtering e.g. "exceptional health"
        for label, (desc, score, baseline) in dimension_results.items():
            diff_pct = ((score - baseline) / baseline) * 100
            if diff_pct > 50:
                level = "exceptional"
            elif diff_pct > 20:
                level = "strong"
            elif diff_pct > 0:
                level = "good"
            elif diff_pct > -20:
                level = "below average"
            else:
                level = "poor"
            metadata[f"{label}_performance"] = level

        # Benefit flags per dimension — for filtering e.g. "positive environmental benefit"
        for dim, label in DIMENSION_OPTIONS.items():
            benefit = row.get(f"{dim}_benefit")
            try:
                b = float(benefit)
                if not pd.isna(b):
                    metadata[f"{label}_benefit"] = "positive" if b > 0 else "negative" if b < 0 else None
            except (ValueError, TypeError):
                metadata[f"{label}_benefit"] = None

        # ---------------------------------------------------------------- #
        # Text — descriptive for embedding                                 #
        # ---------------------------------------------------------------- #
        lines = []

        # Opening summary
        exceptional, strong, good, below, poor = [], [], [], [], []
        for label, (desc, score, baseline) in dimension_results.items():
            diff_pct = ((score - baseline) / baseline) * 100
            if diff_pct > 50:   exceptional.append(label)
            elif diff_pct > 20: strong.append(label)
            elif diff_pct > 0:  good.append(label)
            elif diff_pct > -20:below.append(label)
            else:               poor.append(label)

        all_above = exceptional + strong + good
        all_below = below + poor

        if exceptional and not strong and not good and not below and not poor:
            lines.append(f"Product demonstrates exceptional sustainability performance {country_str} across all measured dimensions.")
        elif all_above and not all_below:
            lines.append(f"Product performs above the market average {country_str} across all measured dimensions.")
        elif all_below and not all_above:
            lines.append(f"Product performs below the market average {country_str} across all measured dimensions.")
        else:
            lines.append(f"Product has mixed sustainability performance {country_str}.")

        # Dimensional breakdown
        parts = [desc for _, (desc, _, _) in dimension_results.items()]
        lines.append(", ".join(parts) + ".")

        # Standout highlights
        if exceptional:
            lines.append(f"Particularly strong in {' and '.join(exceptional)}, far exceeding market benchmarks {country_str}.")
        if poor:
            lines.append(f"Notably underperforms in {' and '.join(poor)} compared to similar TABLE_07 {country_str}.")

        # Benefit summary
        benefit_parts = []
        for dim, label in DIMENSION_OPTIONS.items():
            benefit = row.get(f"{dim}_benefit")
            desc = describe_benefit(benefit, label)
            if desc:
                benefit_parts.append(desc)
        if benefit_parts:
            lines.append(f"Offers: {', '.join(benefit_parts)}.")

        chunks.append({
            "text":     " ".join(lines),
            "metadata": metadata,
        })

    return chunks


def build_all_score_chunks(df: pd.DataFrame, product_col: str = COL_04) -> list[dict]:
    all_chunks = []
    for pid, group in df.groupby(product_col):
        chunks = serialize_product_score_chunks(pid, group)
        all_chunks.extend(chunks)
    return all_chunks

In [ ]:
scores_df = build_product_scores_combined_df(df_24, df_25)
all_score_chunks = build_all_score_chunks(scores_df)

for chunk in all_score_chunks[:2]:
    print("--- METADATA ---")
    print(chunk["metadata"])
    print("--- TEXT ---")
    print(chunk["text"])
    print()

In [ ]:
save_chunks_to_csv(all_score_chunks, "processed_chunks/scores_chunks.csv")

### Table 'product_group_attributes'

In [ ]:
with engine.connect() as connection:
    df_26 = pd.read_sql("SELECT * FROM TABLE_12", connection)
    df_27 = pd.read_sql("SELECT * FROM TABLE_21", connection)
    df_28 = pd.read_sql("SELECT * FROM TABLE_04", connection)
    df_29 = pd.read_sql("SELECT * FROM TABLE_17", connection)

print("df_26 columns:", df_26.columns.tolist())
print("df_26 columns:", len(df_26.columns.tolist()))
print("df_27 columns:", df_27.columns.tolist())
print("df_28 columns:", df_28.columns.tolist())

In [ ]:
def clean_description(text: str) -> str | None:
    """
    Strip HTML tags, extract only the first meaningful sentence(s)
    before boilerplate like 'Impact Criteria', 'Scoring Decisions'.
    """
    if is_empty(str(text)) or pd.isna(text):
        return None

    # Remove all HTML tags
    text = re.sub(r"<[^>]+>", " ", text)
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    # Cut off at boilerplate markers
    for marker in ["Impact Criteria", "Scoring Decisions", "Cut-off Criteria"]:
        idx = text.find(marker)
        if idx != -1:
            text = text[:idx].strip()

    # Clean trailing punctuation artifacts
    text = text.strip().rstrip(".</br>").strip()

    return text if text else None


# Build lookup from main_id name → cleaned description
ATTRIBUTE_DESCRIPTIONS_OPTIONS = {}

attributes_desc_df = df_27[[COL_37, COL_01, COL_38]].drop_duplicates(subset=[COL_37])
for _, row in attributes_desc_df.iterrows():
    name = str(row[COL_01]).strip()
    desc = clean_description(str(row.get(COL_38, "")))
    if desc:
        ATTRIBUTE_DESCRIPTIONS_OPTIONS[name] = desc

for name, desc in ATTRIBUTE_DESCRIPTIONS_OPTIONS.items():
    print(f"\n--- {name} ---")
    print(desc)

In [ ]:
def standardise_decision(text: str) -> str:
    if not text or pd.isna(text):
        return text
    
    t = str(text).strip()
    t = re.sub(r"\s+", " ", t)
    t = t.replace("\u2019", "'").replace("\u00e2\u0080\u0099", "'")
    
    t_lower = t.lower()
    
    # Standardise all "no evidence" variants → canonical form
    if t_lower.startswith("no") and any(phrase in t_lower for phrase in [
        "no evidence", "not enough evidence", "not evaluated"
    ]):
        return "No - No evidence"
    
    # Standardise all "doesn't meet cut-off" variants → canonical form
    if t_lower.startswith("no") and any(phrase in t_lower for phrase in [
        "doesn't meet", "doesnot meet", "does not meet", "cut-off", "cut off"
    ]):
        return "No - Evidence doesn't meet cut-off"
    
    return t

# Apply standardisation before classification
df_26[COL_39] = df_26[COL_39].apply(standardise_decision)
df_26[COL_39].unique()

In [ ]:
# Map key_value to rating labels
KEY_0_VALUE_MAP = {
    (0.0, "No - No evidence")                       : "no data",
    (0.0, "No - Evidence doesn't meet cut-off")     : "poor",
}
KEY_VALUE_MAP = {
    0.5: "fair",
    1.0: "good",
    1.5: "very good",
    2.0: "excellent",
    4.0: "exceptional",
    5.0: "exceptional",
    6.0: "exceptional",
}
df_26[COL_40] = pd.to_numeric(df_26[COL_40], errors="coerce")

def classify_row(row):
    kv  = row[COL_40]
    dec = row[COL_39]

    # Check zero-value cases first using both columns
    if kv == 0.0:
        key_0 = (0.0, dec)
        if key_0 in KEY_0_VALUE_MAP:
            return KEY_0_VALUE_MAP[key_0]
        return "no data"

    # Map by key_value alone for all other cases
    if kv in KEY_VALUE_MAP:
        return KEY_VALUE_MAP[kv]

    return "no data"

df_26[COL_44] = df_26.apply(classify_row, axis=1)
df_26 = df_26.drop(columns=[COL_39, COL_40])

# Sanity check
print(df_26[COL_44].value_counts())
print(f"\nUnclassified: {(df_26['rating'] == 'no data').sum()}")

In [10]:
ATTRIBUTE_DESCRIPTIONS_OPTIONS = json.loads(os.getenv("ATTRIBUTE_DESCRIPTIONS_MAP"))

def preprocess_attributes(df: pd.DataFrame, attributes_df: pd.DataFrame, countries_df: pd.DataFrame, units_df: pd.DataFrame) -> pd.DataFrame:

    # --- Shared helpers ---
    units_lookup = units_df[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).copy()
    units_lookup[COL_03] = pd.to_numeric(units_lookup[COL_03], errors="coerce")
    units_map = units_lookup.set_index(COL_03)[COL_01].to_dict()

    def resolve_unit(uid):
        try:
            return units_map.get(int(uid), None)
        except (ValueError, TypeError):
            return None

    # Resolve attribute name from attributes_df
    attrs = attributes_df[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).rename(
        columns={COL_03: COL_41, COL_01: COL_42}
    )
    df = df.merge(attrs, left_on=COL_43, right_on=COL_41, how="left")

    # Remove rows where there is no data
    df = df[df[COL_44] != "no data"]

    # ----- Attribute 1, 2: Locally made & Reclaimed: data_value is a country_id -----
    country_lookup = countries_df[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).copy()
    country_lookup[COL_03] = pd.to_numeric(country_lookup[COL_03], errors="coerce")

    locally_made_mask = df[COL_42].isin(["Locally made", "Reclaimed"])

    # Convert data_value to numeric for joining
    df.loc[locally_made_mask, COL_45] = pd.to_numeric(
        df.loc[locally_made_mask, COL_46], errors="coerce"
    )

    # Map country_id to country name
    country_map = country_lookup.set_index(COL_03)[COL_01].to_dict()
    df.loc[locally_made_mask, COL_47] = (
        df.loc[locally_made_mask, COL_45].map(country_map)
    )

    # Build "locally made in X" / "reclaimed in X", leave NaN rows as NaN
    def build_location_phrase(row):
        country = row[COL_47]
        if pd.isna(country):
            return None
        if row[COL_42] == "Locally made":
            return f"locally made in {country}"
        if row[COL_42] == "Reclaimed":
            return f"reclaimed in {country}"
        return None

    df.loc[locally_made_mask, COL_46] = (
        df[locally_made_mask].apply(build_location_phrase, axis=1)
    )

    df = df.drop(columns=[COL_45, COL_47])

    # ----------------------------------------------------------------------------------------
    #   Attribute 3, 14: Embodied Energy & Thermal: data_value is numerical, join with unit 
    # ----------------------------------------------------------------------------------------
    EE_OPTIONS = json.loads(os.getenv("RATING_TO_EE_MAP"))
    TB_OPTIONS = json.loads(os.getenv("RATING_TO_TB_MAP"))

    def build_numerical_phrase(row):
        val    = row[COL_46]
        unit   = row[COL_48]
        attr   = row[COL_42]
        rating = row[COL_44]

        if is_empty(str(val)) or pd.isna(val) or val is None:
            return val

        val_clean     = str(val).strip().lstrip(",").strip()
        val_with_unit = f"{val_clean} {unit}".strip() if pd.notna(unit) else val_clean

        if attr == "Low embodied energy":
            band = EE_OPTIONS.get(rating)
            if band:
                return f"{band} of {val_with_unit}"
            return f"low embodied energy of {val_with_unit}"

        if attr == "Thermal barrier":
            band = TB_OPTIONS.get(rating)
            if band:
                return f"{band}, thermal barrier of {val_with_unit}"
            return f"thermal barrier of {val_with_unit}"

        return val_with_unit

    units_lookup = units_df[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).copy()
    units_lookup[COL_03] = pd.to_numeric(units_lookup[COL_03], errors="coerce")
    units_map = units_lookup.set_index(COL_03)[COL_01].to_dict()

    numerical_mask = df[COL_42].isin(["Low embodied energy", "Thermal barrier"])

    df.loc[numerical_mask, COL_49] = pd.to_numeric(
        df.loc[numerical_mask, COL_50], errors="coerce"
    )
    df.loc[numerical_mask, COL_48] = df.loc[numerical_mask, COL_49].map(units_map)

    # Apply phrase builder instead of raw value + unit concatenation
    df.loc[numerical_mask, COL_46] = df[numerical_mask].apply(build_numerical_phrase, axis=1)

    df = df.drop(columns=[COL_49, COL_48])

    # ----------------------------------------------------------------------------------------
    #   Attribute 4: Emboided Carbon
    # ----------------------------------------------------------------------------------------

    GWP_LABELS_OPTIONS_OPTIONS = json.loads(os.getenv("GWP_LABELS_MAP"))
    RTC_BAND_OPTIONS = json.loads(os.getenv("RTC_BAND_MAP"))
    
    lec_mask = df[COL_42] == "Low embodied carbon"    

    def parse_low_embodied_carbon(row):
        data_val  = str(row[COL_46]).strip() if not is_empty(str(row[COL_46])) else ""
        data_unit = str(row[COL_50]).strip()  if not is_empty(str(row[COL_50]))  else ""
        rating    = row[COL_44]

        if not data_val or data_val == "0":
            return None

        unit_name = resolve_unit(data_unit)
        unit_str  = f" {unit_name}" if unit_name else ""

        def band_and_format(val_str, label):
            try:
                fval = float(val_str)
                if fval == 0:
                    return None

                # Keep numeric logic for carbon negative — rating can't capture this
                if fval < 0:
                    band = "carbon negative, carbon sequestering product, carbon sink"
                else:
                    band = RTC_BAND_OPTIONS.get(rating, "low embodied carbon")

                if label:
                    full_label = GWP_LABELS_OPTIONS.get(label, label)
                    polarity = "negative" if fval < 0 else "positive"
                    return f"{full_label} is {polarity} at {fval}{unit_str}, which is {band}"
                else:
                    return f"Embodied carbon is {fval}{unit_str}, which is {band}"

            except ValueError:
                return val_str

        # Comma separated — GWP-Total, GWP-Fossil, GWP-Biogenic
        if "," in data_val:
            parts = data_val.split(",")
            while len(parts) < 3:
                parts.append("")

            gwp_total    = parts[0].strip()
            gwp_fossil   = parts[1].strip()
            gwp_biogenic = parts[2].strip()

            segments = []
            if gwp_total and gwp_total not in ("0", ""):
                seg = band_and_format(gwp_total, "GWP-Total")
                if seg:
                    segments.append(seg)
            if gwp_fossil and gwp_fossil not in ("0", ""):
                seg = band_and_format(gwp_fossil, "GWP-Fossil")
                if seg:
                    segments.append(seg)
            if gwp_biogenic and gwp_biogenic not in ("0", ""):
                seg = band_and_format(gwp_biogenic, "GWP-Biogenic")
                if seg:
                    segments.append(seg)

            return ". ".join(segments) if segments else None

        # Single value — total GWP
        return band_and_format(data_val, "GWP-Total")

    df.loc[lec_mask, COL_46] = df[lec_mask].apply(parse_low_embodied_carbon, axis=1)


    # ----------------------------------------------------------------------------------------
    #   Attribute 5, 6: Rapidly degradable, Rapidly renewable
    # ----------------------------------------------------------------------------------------

    rapid_mask = df[COL_42].isin(["Rapidly degradable", "Rapidly renewable"])

    RTG_OPTIONS = json.loads(os.getenv("RTG_MAP"))
    RTR_OPTIONS = json.loads(os.getenv("RTR_MAP"))

    def parse_rapid(row):
        data_val = str(row[COL_46]).strip() if not is_empty(str(row[COL_46])) else ""
        attr     = str(row[COL_42])
        rating   = row[COL_44]

        if not data_val or data_val == "0":
            return None

        noun        = "degradable" if "degradable" in attr else "renewable"
        rating_map  = RTG_OPTIONS if "degradable" in attr else RTR_OPTIONS

        # Plain text statement — no numeric value to extract
        if not any(c.isdigit() for c in data_val) or "Product" in data_val or "year" in data_val.lower():
            cleaned = re.sub(r"\s+", " ", data_val).strip()
            return f"rapidly {noun}: {cleaned}"

        # Comma separated
        parts = [p.strip() for p in data_val.split(",")]

        if len(parts) <= 2:
            return None

        years_val = parts[1].strip()

        if not years_val or years_val in ("0", ""):
            return None

        try:
            years    = float(years_val)
            template = rating_map.get(rating)
            if years > 0 and template:
                return template.format(y=years)
            return None
        except ValueError:
            return f"rapidly {noun}: {years_val}"

    df.loc[rapid_mask, COL_46] = df[rapid_mask].apply(parse_rapid, axis=1)

    # ----------------------------------------------------------------------------------------
    #   Attribute 7: Recycled content
    # ----------------------------------------------------------------------------------------

    RT_RECYCLED_OPTIONS = json.loads(os.getenv("RT_RECYCLED_MAP"))

    recycled_mask = df[COL_42] == "Recycled content"

    def parse_recycled_content(row):
        data_val = str(row[COL_46]).strip() if not is_empty(str(row[COL_46])) else ""
        rating   = row[COL_44]

        if not data_val or data_val == "0":
            return None

        band = RT_RECYCLED_OPTIONS.get(rating, "")

        def band_and_format(val_str, pre_post):
            try:
                fval = float(val_str)
                if fval == 0:
                    return None
                prefix = f"{band}" if band else ""
                return f"{prefix} {pre_post} recycled content of {fval:.1f}%"
            except ValueError:
                return f"{pre_post} recycled content of {val_str}"

        if "," in data_val:
            parts = data_val.split(",")
            pre  = parts[0].strip()
            post = parts[1].strip() if len(parts) > 1 else ""

            segments = []
            if pre and pre not in ("0", "0.0", ""):
                seg = band_and_format(pre, "pre-consumer")
                if seg:
                    segments.append(seg)
            if post and post not in ("0", "0.0", ""):
                seg = band_and_format(post, "post-consumer")
                if seg:
                    segments.append(seg)

            return "; ".join(segments) if segments else None

        return band_and_format(data_val, "pre-consumer")

    df.loc[recycled_mask, COL_46] = df[recycled_mask].apply(parse_recycled_content, axis=1)
    

    # ----------------------------------------------------------------------------------------
    #   Attribute 8: End of life plan
    # ----------------------------------------------------------------------------------------
    eol_mask = df[COL_42] == "End of life plan"

    def parse_eol(row):
        data_val = str(row[COL_46]).strip() if not is_empty(str(row[COL_46])) else ""

        if not data_val or data_val == "0":
            return None

        # Clean up whitespace
        cleaned = re.sub(r"\s+", " ", data_val).strip()
        return f"{cleaned}" if cleaned else None

    df.loc[eol_mask, COL_46] = df[eol_mask].apply(parse_eol, axis=1)


    # ----------------------------------------------------------------------------------------
    #   Attribute 9: Durable
    # ----------------------------------------------------------------------------------------
    RTS_OPTIONS = json.loads(os.getenv("RTS_MAP"))

    durable_mask = df[COL_42] == "Durable"

    def parse_durable(row):
        data_val = str(row[COL_46]).strip() if not is_empty(str(row[COL_46])) else ""
        rating   = row[COL_44]

        if not data_val or data_val == "0":
            return None

        # Lifetime is unambiguous — keep as is
        if data_val.lower() == "lifetime":
            return "very long service life, expected to last a lifetime"

        band = RTS_OPTIONS.get(rating, "")

        # Plain text with no numeric value
        if any(c.isalpha() for c in data_val):
            return f"{band}: {data_val}" if band else f"service life: {data_val}"

        try:
            fval = float(data_val)
            return f"{band} of {fval:.0f} years" if band else f"service life of {fval:.0f} years"
        except ValueError:
            return f"{band}: {data_val}" if band else f"service life: {data_val}"

    df.loc[durable_mask, COL_46] = df[durable_mask].apply(parse_durable, axis=1)


    # ----------------------------------------------------------------------------------------
    #   Attribute 10: Versatile
    # ----------------------------------------------------------------------------------------
    versatile_mask = df[COL_42] == "Versatile"

    # Canonical clean descriptions for each option code
    V_OPTIONS = json.loads(os.getenv("V_OPTIONS"))

    def parse_versatile(row):
        data_val = str(row[COL_46]).strip() if not is_empty(str(row[COL_46])) else ""

        if not data_val or data_val == "0":
            return None

        # Extract option codes (a-e) from the text
        codes_found = re.findall(r"\(([a-eA-E])\)", data_val)
        codes_found = list(dict.fromkeys([c.lower() for c in codes_found]))  # deduplicate, preserve order

        if codes_found:
            descriptions = [V_OPTIONS[c] for c in codes_found if c in V_OPTIONS]
            if descriptions:
                return "This product " + "; ".join(descriptions) + "."

        # No codes found — clean up and return as-is
        cleaned = re.sub(r"\([a-eA-E]\)\s*", "", data_val)
        cleaned = re.sub(r"\s+", " ", cleaned).strip().rstrip(".")
        return cleaned if cleaned else None

    df.loc[versatile_mask, COL_46] = df[versatile_mask].apply(parse_versatile, axis=1)


    # ----------------------------------------------------------------------------------------
    #   Attribute 11: Low toxicity
    # ----------------------------------------------------------------------------------------
    low_tox_mask = df[COL_42] == "Low toxicity"

    parts = []

    def parse_low_toxicity(row):
        data_val  = str(row[COL_46]).strip() if not is_empty(str(row[COL_46])) else ""
        data_unit = str(row[COL_50]).strip()  if not is_empty(str(row[COL_50]))  else ""

        # Split into parts
        val_parts  = data_val.split(",")  if data_val  else []
        unit_parts = data_unit.split(",") if data_unit else []

        # Pad to 3 positions
        while len(val_parts)  < 3: val_parts.append("")
        while len(unit_parts) < 3: unit_parts.append("")

        voc_val   = val_parts[0].strip()
        form_val  = val_parts[1].strip()
        statement = val_parts[2].strip()

        voc_unit  = unit_parts[0].strip()
        form_unit = unit_parts[1].strip()

        voc_unit_name  = resolve_unit(voc_unit)
        form_unit_name = resolve_unit(form_unit)

        # VOC
        if voc_val and voc_val not in ("0", ""):
            if voc_unit_name:
                parts.append(f"VOC emissions of {voc_val} {voc_unit_name}")
            else:
                parts.append(f"VOC emissions of {voc_val}")

        # Formaldehyde
        if form_val and form_val not in ("0", ""):
            if form_unit_name:
                parts.append(f"formaldehyde emissions of {form_val} {form_unit_name}")
            else:
                parts.append(f"formaldehyde emissions of {form_val}")

        # General statement
        if statement and statement not in ("0", ""):
            cleaned = re.sub(r"(?i)Manufacturer states that the\s*", "", statement).strip()
            parts.append(cleaned)

        if parts:
            return "; ".join(parts)
        return None

    df.loc[low_tox_mask, COL_46] = df[low_tox_mask].apply(parse_low_toxicity, axis=1)

    # ----------------------------------------------------------------------------------------
    #   Attribute 12: Moisture Balancing: single numeric value with unit, or plain text statement
    # ----------------------------------------------------------------------------------------

    RTM_OPTIONS = json.loads(os.getenv("RTM_MAP"))

    moisture_mask = df[COL_42] == "Moisture balancing"

    def parse_moisture(row):
        data_val  = str(row[COL_46]).strip() if not is_empty(str(row[COL_46])) else ""
        data_unit = str(row[COL_50]).strip()  if not is_empty(str(row[COL_50]))  else ""
        rating    = row[COL_44]

        if not data_val or data_val == "0":
            return None

        band = RTM_OPTIONS.get(rating, "")

        try:
            fval      = float(data_val)
            unit_name = resolve_unit(data_unit)
            prefix    = f"{band}, " if band else ""
            if unit_name:
                return f"{prefix}moisture resistance value of {fval} {unit_name}"
            return f"{prefix}moisture resistance value of {fval}"
        except ValueError:
            cleaned = re.sub(r"(?i)manufacturer states that the product is\s*", "", data_val).strip()
            return f"{band}, {cleaned.lower()}" if band else cleaned.lower()

    df.loc[moisture_mask, COL_46] = df[moisture_mask].apply(parse_moisture, axis=1)

    # ----------------------------------------------------------------------------------------
    #   Attribute 13: Acoustic Regulator
    # ----------------------------------------------------------------------------------------

    RTIS_OPTIONS = json.loads(os.getenv("RTIS_MAP"))
    RTAS_OPTIONS = json.loads(os.getenv("RTAS_MAP"))
    RTN_OPTIONS = json.loads(os.getenv("RTN_MAP"))

    acoustic_mask = df[COL_42] == "Acoustic regulator"

    def parse_acoustic(row):
        data_val  = str(row[COL_46]).strip() if not is_empty(str(row[COL_46])) else ""
        data_unit = str(row[COL_50]).strip()  if not is_empty(str(row[COL_50]))  else ""
        rating    = row[COL_44]

        if not data_val or data_val == "0":
            return None

        val_parts  = data_val.split(",")  if "," in data_val else [data_val]
        unit_parts = data_unit.split(",") if "," in data_unit else []

        while len(val_parts)  < 4: val_parts.append("")
        while len(unit_parts) < 3: unit_parts.append("")

        impact    = val_parts[0].strip()
        airborne  = val_parts[1].strip()
        nrc       = val_parts[2].strip()
        statement = val_parts[3].strip()

        impact_unit   = unit_parts[0].strip()
        airborne_unit = unit_parts[1].strip()
        nrc_unit      = unit_parts[2].strip()

        parts = []

        if impact and impact not in ("0", ""):
            try:
                fval      = float(impact)
                band      = RTIS_OPTIONS.get(rating, "")
                unit_name = resolve_unit(impact_unit)
                unit_str  = f" {unit_name}" if unit_name else ""
                parts.append(f"{band} of {fval}{unit_str}" if band else f"impact sound of {fval}{unit_str}")
            except ValueError:
                parts.append(f"impact sound: {impact}")

        if airborne and airborne not in ("0", ""):
            try:
                fval      = float(airborne)
                band      = RTAS_OPTIONS.get(rating, "")
                unit_name = resolve_unit(airborne_unit)
                unit_str  = f" {unit_name}" if unit_name else ""
                parts.append(f"{band}, airborne sound reduction of {fval}{unit_str}" if band else f"airborne sound reduction of {fval}{unit_str}")
            except ValueError:
                parts.append(f"airborne sound: {airborne}")

        if nrc and nrc not in ("0", ""):
            try:
                fval      = float(nrc)
                band      = RTN_OPTIONS.get(rating, "")
                unit_name = resolve_unit(nrc_unit)
                unit_str  = f" {unit_name}" if unit_name else ""
                parts.append(f"{band}, NRC of {fval}{unit_str}" if band else f"NRC of {fval}{unit_str}")
            except ValueError:
                parts.append(f"NRC: {nrc}")

        if statement and statement not in ("0", ""):
            parts.append(statement)

        return "; ".join(parts) if parts else None

    df.loc[acoustic_mask, COL_46] = df[acoustic_mask].apply(parse_acoustic, axis=1)


    # ----------------------------------------------------------------------------------------
    #   Attribute 15: Fire resistance
    # ----------------------------------------------------------------------------------------
    fire_mask = df[COL_42] == "Fire resistance"

    FC_OPTIONS = json.loads(os.getenv("FC_MAP"))

    def parse_fire(row):
        data_val = str(row[COL_46]).strip() if not is_empty(str(row[COL_46])) else ""

        if not data_val or data_val == "0":
            return None

        cleaned = re.sub(r"\s+", " ", data_val).strip()

        # Look up full description, fall back to cleaned text
        full_desc = FC_OPTIONS.get(cleaned, cleaned)
        return f"{full_desc}"

    df.loc[fire_mask, COL_46] = df[fire_mask].apply(parse_fire, axis=1)

    return df

def build_all_attribute_chunks(df: pd.DataFrame, product_col: str = COL_04) -> list[dict]:
    all_chunks = []

    for pid, group in df.groupby(product_col):
        lines      = []
        attr_names = []

        for _, row in group.iterrows():
            attr_name = row.get(COL_42)
            data_val  = row.get(COL_46)

            if is_empty(str(attr_name)) or pd.isna(attr_name):
                continue
            if is_empty(str(data_val)) or pd.isna(data_val):
                continue

            attr_names.append(str(attr_name))
            description = ATTRIBUTE_DESCRIPTIONS_OPTIONS.get(str(attr_name).strip())
            if description:
                lines.append(f"{description}: {str(data_val).strip()}.")
            else:
                lines.append(f"{str(data_val).strip()}.")

        if not lines:
            continue

        all_chunks.append({
            "text": "\n".join(lines),
            "metadata": {
                COL_04:      str(pid),
                "doc_type":        "attributes",
                "attribute_names": ", ".join(set(attr_names)),
            }
        })

    return all_chunks

In [ ]:
attributes_preprocessed_df = preprocess_attributes(
    df_26.copy(), df_27, df_29, df_28
)
attribute_chunks = build_all_attribute_chunks(attributes_preprocessed_df)

sample_id = 175
print(f"Total attribute chunks: {len(attribute_chunks)}")
print(f"\n--- METADATA ---")
print(attribute_chunks[sample_id]["metadata"])
print(f"--- TEXT ---")
print(attribute_chunks[sample_id]["text"])

In [ ]:
save_chunks_to_csv(attribute_chunks, "processed_chunks/attribute_chunks.csv")

### Table 'TABLE_07'

In [ ]:
with engine.connect() as connection:
    TABLE_07_df = pd.read_sql("SELECT * FROM TABLE_07", connection)
    TABLE_22_df = pd.read_sql("SELECT * FROM TABLE_22", connection)
    TABLE_23_df = pd.read_sql("SELECT * FROM TABLE_23", connection)
    TABLE_24_df = pd.read_sql("SELECT * FROM TABLE_24", connection)
    TABLE_25_df = pd.read_sql("SELECT * FROM TABLE_25", connection)
    TABLE_26_df = pd.read_sql("SELECT * FROM TABLE_26", connection)
    units_df = pd.read_sql("SELECT * FROM units", connection)
    countries_df = pd.read_sql("SELECT * FROM countries", connection)
    TABLE_27_df = pd.read_sql("SELECT * FROM TABLE_27", connection)
    TABLE_28_df = pd.read_sql("SELECT * FROM TABLE_28", connection)
    TABLE_29_df = pd.read_sql("SELECT * FROM TABLE_29", connection)
    TABLE_30_df = pd.read_sql("SELECT * FROM TABLE_30", connection)
    TABLE_30i_df = pd.read_sql("SELECT * FROM TABLE_30i", connection)
    TABLE_30n_df = pd.read_sql("SELECT * FROM TABLE_30n", connection)
    TABLE_30r_df = pd.read_sql("SELECT * FROM TABLE_30r", connection)
    TABLE_31_df = pd.read_sql("SELECT * FROM TABLE_31", connection)
    TABLE_31cs_df = pd.read_sql("SELECT * FROM TABLE_31cs", connection)
    TABLE_31ci_df = pd.read_sql("SELECT * FROM TABLE_31ci", connection)
    TABLE_32_df = pd.read_sql("SELECT * FROM TABLE_32", connection)
    TABLE_33_df = pd.read_sql("SELECT * FROM TABLE_33", connection)

print("TABLE_07_df columns:", TABLE_07_df.columns.tolist())
print("TABLE_07_df columns count:", len(TABLE_07_df.columns.tolist()))
print("TABLE_22_df columns:", TABLE_22_df.columns.tolist())
print("TABLE_23_df columns:", TABLE_23_df.columns.tolist())
print("TABLE_24_df columns:", TABLE_24_df.columns.tolist())
print("TABLE_25_df columns:", TABLE_25_df.columns.tolist())
print("TABLE_26_df columns:", TABLE_26_df.columns.tolist())

In [ ]:
def build_TABLE_07_combined_df(TABLE_07_df, TABLE_22_df, TABLE_23_df, 
                                TABLE_24_df, TABLE_25_df, TABLE_26_df):
    
    # Keep relevant columns
    df = TABLE_07_df[[COL_03, "product_type_detail_id", COL_01, "brand_name", "residential_commercial", "unit", "density", 
                      "thickness", "thickness_unit", "application", "product_type_tag", "project_name",
                      "descriptions", "key_advantages", "limitations", "is_reclaimed", 
                      "TABLE_27", "special_area", "TABLE_29", "breeam_features",
                      "breeami_features", "breeamn_features", "breeamr_features", "leedci_features", "leedcs_features", "leed_features",
                      "sdg_features"]]
    
    print("TABLE_07_df columns:", df.columns.tolist())
    print("TABLE_07_df columns:", len(df.columns.tolist()))
    
    # -----------------------------------
    #    Product details 
    # -----------------------------------
    TABLE_22 = TABLE_22_df[[COL_03, COL_01, "product_type_id", "category_id", "family_id", "group_id"]].drop_duplicates(subset=[COL_03]).rename(
        columns={COL_01: "product_detail_name"}
    )
    TABLE_23 = TABLE_23_df[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).rename(
        columns={COL_03: "pt_id", COL_01: "product_type_name"}
    )
    TABLE_24 = TABLE_24_df[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).rename(
        columns={COL_03: "cat_id", COL_01: "category_name"}
    )
    TABLE_25 = TABLE_25_df[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).rename(
        columns={COL_03: "fam_id", COL_01: "family_name"}
    )
    TABLE_26 = TABLE_26_df[[COL_03, COL_01]].drop_duplicates(subset=[COL_03]).rename(
        columns={COL_03: "grp_id", COL_01: "group_name"}
    )

    # Merge TABLE_22
    df = df.merge(
        TABLE_22,
        left_on="product_type_detail_id",
        right_on=COL_03,
        how="left"
    ).drop(columns=[COL_03])

    # Merge product_type name
    df = df.merge(
        TABLE_23,
        left_on="product_type_id",
        right_on="pt_id",
        how="left"
    ).drop(columns=["pt_id", "product_type_id"])

    # Merge category name
    df = df.merge(
        TABLE_24,
        left_on="category_id",
        right_on="cat_id",
        how="left"
    ).drop(columns=["cat_id", "category_id"])

    # Merge family name
    df = df.merge(
        TABLE_25,
        left_on="family_id",
        right_on="fam_id",
        how="left"
    ).drop(columns=["fam_id", "family_id"])

    # Merge group name
    df = df.merge(
        TABLE_26,
        left_on="group_id",
        right_on="grp_id",
        how="left"
    ).drop(columns=["grp_id", "group_id"])

    # -----------------------------------
    #    Certification details 
    # -----------------------------------
    # Define the mapping: (TABLE_07_df column, lookup dataframe)
    column_lookup_map = [
        ("special_area",       TABLE_28_df),
        ("TABLE_29",      TABLE_29_df),
        ("breeam_features",    TABLE_30_df),
        ("breeami_features",   TABLE_30i_df),
        ("breeamn_features",   TABLE_30n_df),
        ("breeamr_features",   TABLE_30r_df),
        ("leed_features",      TABLE_31_df),
        ("leedcs_features",    TABLE_31cs_df),
        ("leedci_features",    TABLE_31ci_df),
        ("sdg_features",       TABLE_32_df),
    ]

    def resolve_ids(cell, id_to_name):
        if pd.isna(cell) or cell == "":
            return cell
        ids = str(cell).split(",")
        names = []
        for i in ids:
            i = i.strip().strip("'\"")  # strip spaces and any stray quotes
            if i == "":
                continue
            try:
                names.append(id_to_name.get(int(i), i))
            except ValueError:
                names.append(i)  # leave as-is if it can't be cast to int
        return ", ".join(names)

    for col, lookup_df in column_lookup_map:
        id_to_name = lookup_df.set_index(COL_03)[COL_01].to_dict()
        df[col] = df[col].apply(lambda x: resolve_ids(x, id_to_name))


    # -----------------------------------
    #    Brand Name - Company Info
    # -----------------------------------
    companies = TABLE_33_df[[COL_03, COL_01, "display_name", "country"]].drop_duplicates(subset=[COL_03]).rename(
        columns={COL_03: "company_id", COL_01: "company_name", "display_name": "company_display_name", "country": "company_country"}
    )
    countries = countries_df[[COL_03,COL_01]].drop_duplicates(subset=[COL_03]).rename(
        columns={COL_03: COL_28, COL_01: COL_29}
    )
    # Cast both to the same type
    df["brand_name"] = df["brand_name"].astype(str)
    companies["company_id"] = companies["company_id"].astype(str)
    
    df = df.merge(
        companies,
        left_on="brand_name",
        right_on="company_id",
        how="left"
    ).drop(columns=["brand_name", "company_id"])

    df["company_country"] = df["company_country"].astype(str)
    countries[COL_28] = countries[COL_28].astype(str)

    df = df.merge(
        countries,
        left_on="company_country",
        right_on=COL_28,
        how="left"
    ).drop(columns=["company_country", COL_28])

    # -----------------------------------
    #    Density
    # -----------------------------------
    # Normalise units
    unit_map = {
        "kg/m2": "kg/m²",
        "Kg/m2": "kg/m²",
        "kg/m²": "kg/m²",
        "kg/m3": "kg/m³",
        "Kg/m3": "kg/m³",
        "kg/m³": "kg/m³",
    }

    df["units_clean"] = df["unit"].str.strip().map(unit_map)  # empty strings and unknowns become NaN

    # Combine density + unit, only when both are present
    df["density_with_unit"] = df.apply(
        lambda row: f"density of {row['density']} {row['units_clean']}"
        if pd.notna(row["density"]) and pd.notna(row["units_clean"])
        else None,
        axis=1
    )

    df = df.drop(columns=["unit","units_clean","density"])

    # -----------------------------------
    #    Thickness
    # -----------------------------------
    # Build unit id -> name lookup, excluding "No Unit" (id=0)
    unit_id_to_name = units_df[units_df[COL_03] != 0].set_index(COL_03)[COL_01].to_dict()

    # Clean thickness_unit: map IDs to names, empty strings and "No Unit" become NaN
    df["thickness_unit_clean"] = (
        df["thickness_unit"]
        .str.strip()
        .replace("", None)
        .apply(lambda x: 
            None if pd.isna(x)
            else unit_id_to_name.get(int(x)) if x.isdigit()  # it's an ID, look it up
            else x if x in unit_id_to_name.values()          # it's already a name, keep it
            else None                                          # unrecognised, ignore
        )
    )

    # Combine thickness + unit, only when both are present
    df["thickness_with_unit"] = df.apply(
        lambda row: f"thickness of {row['thickness']} {row['thickness_unit_clean']}"
        if pd.notna(row["thickness"]) and pd.notna(row["thickness_unit_clean"])
        else None,
        axis=1
    )
    
    df = df.drop(columns=["thickness", "thickness_unit", "thickness_unit_clean"])

    # -----------------------------------
    #    Reclaimed/New
    # -----------------------------------
    # Normalise units
    reclaimed_map = {
        "New": "new",
        "new": "new",
        "reclaimed": "reclaimed"
    }
    df["is_reclaimed"] = df["is_reclaimed"].str.strip().map(reclaimed_map)

    return df


TABLE_07_combined_df = build_TABLE_07_combined_df(
    TABLE_07_df, TABLE_22_df, TABLE_23_df,
    TABLE_24_df, TABLE_25_df, TABLE_26_df
)
TABLE_07_combined_df.head(1)
print("TABLE_07_combined_df columns:", TABLE_07_combined_df.columns.tolist())
print("TABLE_07_combined_df columns:", len(TABLE_07_combined_df.columns.tolist()))

In [ ]:
CHUNK_COLS = {
    "core":        [COL_01, "product_type_tag", "product_detail_name", "product_type_name", "category_name", "family_name", "descriptions"],
    "performance": ["key_advantages", "limitations"],
    "compliance":  ["breeam_features", "breeami_features", "breeamn_features", "breeamr_features", "leed_features", "leedci_features", "leedcs_features", "sdg_features"],
    "properties":    ["residential_commercial", COL_29, "company_display_name", "density_with_unit", "thickness_with_unit"],
}

def build_chunk_text(row, cols):
    parts = []
    seen  = set()  # deduplicate across columns

    for col in cols:
        if col not in row:
            continue
        val = str(row[col]).strip()
        if not val or val.lower() == "nan":
            continue
        # Each cell may contain comma-separated names — split and dedup
        for item in val.split(","):
            item = item.strip()
            if item and item not in seen:
                seen.add(item)
                parts.append(item)

    return ", ".join(parts)

def build_chunks_for_row(row, chunk_cols):
    chunks = {}
    for chunk_type, cols in chunk_cols.items():
        text = build_chunk_text(row, cols)
        if not text:
            continue
        chunks[chunk_type] = {
            "metadata": {COL_04: row[COL_03], "doc_type": chunk_type},
            "text": text,
        }
    return chunks

# collect chunks per type
all_chunks = {chunk_type: [] for chunk_type in CHUNK_COLS}

for _, row in TABLE_07_combined_df.iterrows():
    chunks = build_chunks_for_row(row, CHUNK_COLS)
    for chunk_type, chunk in chunks.items():
        all_chunks[chunk_type].append(chunk)

# save each chunk type as its own csv
for chunk_type, chunks in all_chunks.items():
    filepath = f"processed_chunks/product_{chunk_type}_chunks.csv"
    save_chunks_to_csv(chunks, filepath)

In [ ]:
for chunk_type, chunks in all_chunks.items():
    print("FOR", chunk_type)
    print(f"\n--- METADATA ---")
    print(chunks[3]["metadata"])
    print(f"--- TEXT ---")
    print(chunks[3]["text"])